# 静态 Shape 执行流程 — 整图下沉

在前面的章节中，我们已经掌握了如何构图、编译并执行模型。从本节开始，我们从「执行性能」的视角深入：**同一张静态 Shape 模型，在 Device 上是怎么被调度执行的？为什么静态 Shape 能跑得更快？用户又该如何把这份性能收益拿到手？**

答案的核心是 **模型下沉调度（整图下沉）**。本节从用户动作出发，讲清整图下沉的机制、生效条件、配套的 Tiling 下沉，以及如何用 profiling 验证收益。

本节学习大纲如下：

- 为什么静态 Shape 能整图下沉：Host 调度 vs 下沉调度
- 整图下沉机制（用户视角）：加载阶段与执行阶段
- 下沉的生效条件与开关
- Tiling 下沉：再榨干一层 Host 开销
- 下沉收益验证：用 msprof / profiling 对比时序
- 小结

## 1. 为什么静态 Shape 能整图下沉

AI 模型运行需要 Host（CPU）与 Device（NPU）协同——Host 擅长复杂逻辑控制，Device 擅长高并行计算。二者之间的交互越频繁，性能损耗越大。静态 Shape 模型由于所有 Tensor 的 shape 在编译期就已确定，GE 可以把「下发算子」这件事从「每次执行都做」变成「加载时做一次」，这就是整图下沉。

### 1.1 Host 逐算子调度：何时会成为 Host Bound

最朴素的执行方式是：Host CPU 按编译后的执行计划，把可执行节点对应的 Task / Kernel Launch **依次**下发到 Device，NPU 再从执行流中拉取 Task 执行。受图融合和 Lowering 影响，源码算子与最终 Task 并非一一对应。

<p align="left"><img src="./images/host_dispatch.svg" alt="Host 调度：逐算子下发" width="35%"></p>

问题在于：模型在训练 / 推理中会运行**多次**，每次运行都触发 Host 遍历可执行节点并逐项 Launch 对应 Task。当单 Task 计算很快、Task 数量又多时，Host 的下发速度会跟不上 Device 的计算速度，Device 出现空泡等待——这就是典型的 **Host Bound（Host 受限）**。

### 1.2 下沉调度：加载时整图下沉，执行时一个 Task 触发

对于全图满足静态执行条件的模型，编译期可以确定各 Tensor 的 shape，结合内存复用算法完成模型级内存编排，并提前完成**不依赖运行时输入数据**的 Tiling 等 Host 侧计算。因此 GE 提供了 **静态图下沉调度模式**：让编译生成的 Task 在**加载阶段**提前以整图形式下发到 Device；**执行阶段**只需在 Host 侧下发**一个模型执行 Task**，即可触发整张图在 Device 上自主调度执行。对于 Tiling 依赖输入数据（`tiling_depend`）的算子，需要满足第 4 节的 Tiling 下沉条件，否则可能被划入动态执行路径。

<p align="left"><img src="./images/graph_sink.svg" alt="整图下沉：一次加载多次执行" width="40%"></p>

> 这与 AscendIR 章节里「静态图执行阶段 Host 仅下发一个模型 Task」的结论一脉相承——本节把它落到「用户怎么开启、怎么验证」的实操层面。

## 2. 整图下沉机制（用户视角）

模型下沉调度分为**模型加载**和**模型执行**两个阶段。理解这两个阶段，才能解释下沉的开销结构与收益来源。

### 2.1 模型加载（一次性）

模型加载阶段会遍历**编译生成的 Task**，并将其整体预分发到 Device 的执行流上，**区别在于下发到流上不立即执行**。加载是一次性动作，但发生时机取决于接口路径：离线 ACL 路径在 `aclmdlLoadFromFile` 等加载接口中显式完成；在线 GeSession 路径则可能在首次 `RunGraph` 前后完成编译与加载，不能统一表述为“首次模型执行时加载”。

加载阶段 GE 在内部完成的关键工作（用户无需手工干预，了解即可）：

| 加载子步骤 | 作用 |
| --- | --- |
| 内存编排 | 为 Feature Map、权重、输入输出分配设备内存，并完成内存复用规划 |
| 权重 / 变量搬运 | 把权重、Variable 数据搬运到 Device HBM |
| 任务下沉（Task Sink） | 把编译期生成的所有 Task 预先分发到设备流，执行时无需再做 Kernel Launch |
| 零拷贝映射 | 为 Data / NetOutput 节点配置零拷贝内存映射（详见 [4.4 静态 Shape 执行优化技术](./04.04_static_shape_optimization.ipynb)） |

> **Task Sink 的意义**：把编译期生成的所有 Task 预置到 Device，执行时只需触发一次设备侧执行，消除了 Host 侧逐算子的 Kernel 启动开销——这是静态执行器高性能的核心保障。

### 2.2 模型执行（可多次）

加载完成后，像下发单算子 Task 一样向执行流下发**一个模型执行 Task**，NPU 调度到该 Task 时执行模型中所有 Task。如需多次运行模型，仅需多次下发这个模型执行 Task。

每次模型下发时，支持**更新 Feature Map 内存地址和输入输出内存地址**。如果地址发生了更新，会在「模型下沉头开销」里完成模型内算子相关地址的刷新。

### 2.3 时序对比：头开销 + 潜在 E2E 收益

把 Host 逐算子调度和下沉调度的时序放在一起对比，可以直观看出收益来源：

```
Host 逐算子调度（当下发成为瓶颈时表现为 Host Bound）：
Host : |下发1| |下发2| |下发3| ......            （下发节奏受 Host 限制）
Dev  :    |算1|   |算2|   |算3| ...  ← 算子间存在等待 Host 的空泡

模型下沉调度（加载已完成，执行只下发 1 个 Task）：
Host : |模型执行触发与头开销|               （不再逐算子下发；同步调用线程仍需等待）
Dev  :        |算1|算2|算3|...|算N|         ← 减少由 Host 下发不及时造成的空泡
```

几个关键结论：

- 下沉执行的开始仍有 **模型执行头开销**，其中包含必要的地址刷新。
- 下沉通常能降低 Host 调度占比较高场景的 **端到端（E2E）耗时**，但收益并非必然，必须通过稳定态实测确认。
- **头开销越小、单步 Host 下发占比越高（即越是 Host Bound），下沉带来的性能提升幅度越大**。反之，如果模型本身就是 Device Bound（算子计算很重、Host 下发不是瓶颈），下沉收益会相对有限。

> 记住一个原则：**下沉是把「每次执行时多次下发编译后 Task / Kernel Launch」摊销成「加载时的一次整图 Task 预分发 + 每次执行的一次模型 Task 触发」。**

### 2.4 动手实践：实测“首次编译加载、多次整图执行”

下面用 ES 构建一张固定 `[512, 512]` 的多算子图，交给同一个 GE Session 在线编译并在 0 号 NPU 上重复执行。单元分别统计首次 `AddGraph + RunGraph`，随后执行 5 次不计时 warm-up，再统计 20 次稳定运行的端到端时间，并对首末输出做数值校验。为使实验聚焦调度开销，示例显式设置 `precision_mode_v2=origin`，避免默认 FP16 降精度使 FP32 对拍受量化误差干扰。

这些是真实 NPU/GE Session 时间，不是教学成本模型；其中仍包含 Python 调用与结果返回等开销。该 Add/Relu 示例只能展示“首次编译加载”和“重复执行”的耗时差异，**不能单独证明 Task Sink 或 Tiling 下沉已经生效**；若要判断实际执行路径及 Device 时间占比，仍应按第 5 节采集 profiling timeline。

> **耗时提示**：第 1 步包含静态图首次编译与加载，在 CANN 9.0 环境中可能需要数十秒或更长；第 2 步执行 warm-up，第 3 步才进行 20 次稳定计时。


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
# === 真机运行：静态图首次编译/加载与稳定 NPU 执行计时 ===
import time

import numpy as np
from ge.es.graph_builder import GraphBuilder
from ge.es.nn import Relu
from ge.ge_global import GeApi
from ge.graph import Tensor
from ge.graph.types import DataType, Format
from ge.session import Session

DEVICE_ID = 0
GRAPH_ID = 1
SHAPE = [512, 512]
STAGES = 12
WARMUP_RUNS = 5
STABLE_RUNS = 20

# 固定 shape、固定拓扑：每层 Add 后接 Relu。
builder = GraphBuilder("StaticSinkTimingGraph")
x = builder.create_input(
    index=0, name="input_x", data_type=DataType.DT_FLOAT, shape=SHAPE
)
out = x
for _ in range(STAGES):
    out = Relu(out + 0.25)
builder.set_graph_output(out, 0)
graph = builder.build_and_reset()

rng = np.random.default_rng(0)
input_array = rng.normal(size=SHAPE).astype(np.float32)
expected = input_array.copy()
for _ in range(STAGES):
    expected = np.maximum(expected + 0.25, 0)

input_tensor = Tensor(
    input_array.reshape(-1).tolist(),
    None,
    DataType.DT_FLOAT,
    Format.FORMAT_ND,
    SHAPE,
)

ge_api = GeApi()
ge_api.ge_initialize({
    "ge.exec.deviceId": str(DEVICE_ID),
    "ge.graphRunMode": "0",
    "ge.exec.precision_mode_v2": "origin",
})
session = None
first_outputs = None
last_outputs = None
try:
    session = Session()

    print("[INFO] Step 1/3: 首次编译、加载并执行静态图")
    started = time.perf_counter()
    session.add_graph(GRAPH_ID, graph)
    first_outputs = session.run_graph(GRAPH_ID, [input_tensor])
    first_ms = (time.perf_counter() - started) * 1e3
    first_actual = np.asarray(first_outputs[0].data, dtype=np.float32)
    np.testing.assert_allclose(first_actual, expected, rtol=1e-5, atol=1e-5)

    print("[INFO] Step 2/3: 执行 {} 次不计时 warm-up".format(WARMUP_RUNS))
    for _ in range(WARMUP_RUNS):
        last_outputs = session.run_graph(GRAPH_ID, [input_tensor])

    print("[INFO] Step 3/3: 重复执行已加载的静态图并计时")
    stable_ms = []
    for _ in range(STABLE_RUNS):
        started = time.perf_counter()
        last_outputs = session.run_graph(GRAPH_ID, [input_tensor])
        stable_ms.append((time.perf_counter() - started) * 1e3)

    last_actual = np.asarray(last_outputs[0].data, dtype=np.float32)
    np.testing.assert_allclose(last_actual, expected, rtol=1e-5, atol=1e-5)

    print("首次 AddGraph + 编译/加载 + 执行：{:.3f} ms".format(first_ms))
    print("稳定态 {} 次执行：平均 {:.3f} ms，P50 {:.3f} ms，P95 {:.3f} ms，最小 {:.3f} ms".format(
        STABLE_RUNS,
        float(np.mean(stable_ms)),
        float(np.median(stable_ms)),
        float(np.percentile(stable_ms, 95)),
        float(np.min(stable_ms)),
    ))
    print("[OK] 同一静态模型已在 NPU 上重复执行，首末结果均正确")
finally:
    first_outputs = None
    last_outputs = None
    input_tensor = None
    # 释放 Session 引用，由 Session 析构统一释放图资源。
    session = None
    ge_api.ge_finalize()


## 3. 下沉的生效条件与开关

### 3.1 生效的前提条件

整图下沉并非任意模型都能享受，核心前提是 **静态 Shape**：

| 条件 | 说明 |
| --- | --- |
| 静态 Shape | 所有输入、输出及中间 Tensor 的 shape 在多次执行中固定不变（编译期完全确定）。这是能整图下沉的根本前提 |
| 编译期可完成内存编排 | shape 固定 → 可完成模型级内存复用与编排 |
| 编译期可完成主要 Host 计算 | shape 固定时，不依赖运行时输入数据的 Tiling 参数可提前算好；`tiling_depend` 算子属于例外，见第 4 节 |

> 需要区分“整图”和“子图”：包含 Unknown Shape 节点的图不能作为一个完整静态模型整体 Task Sink；但 GE 可以把图划分为 Known Shape 与 Unknown Shape 子图，满足条件的 Known Shape 子图仍可使用静态执行器和 Task Sink，动态部分则走动态执行路径（下一节 4.3 详解）。

### 3.2 用户怎么得到下沉调度

对绝大多数用户而言，**没有一个需要单独打开的通用 Task Sink 开关**。当整个编译结果满足 Known Shape / 静态执行条件时，GE 的静态执行器（Known Shape Executor）会默认走整图下沉路径。固定输入维度是重要入口条件，但还要确认中间 Tensor shape、数据依赖 Tiling、算子能力及最终图划分结果：

- **离线路径（ATC + ACL）**：用 `atc` 指定固定输入维度编译 `.om`（`--input_shape` 中不含 `-1`），再用 `aclmdlLoadFromFile` + `aclmdlExecute` 加载执行；仍需结合编译日志或 profiling 确认实际图划分。
- **在线路径（GeSession）**：构图时为 Data 节点设置固定 shape，走 `AddGraph → CompileGraph → RunGraph`，并确认编译后的整图或目标子图进入 Known Shape 执行路径。

```shell
# 离线：固定输入维度，为静态执行与 Task Sink 提供入口条件
atc --model=resnet50.onnx \
    --framework=5 \
    --output=resnet50_static \
    --input_shape="input:1,3,224,224" \
    --soc_version=Ascend910B1
```

> 要点：**静态 Shape 是整图下沉的核心前提，但不是“写死输入 shape 就必然整图下沉”的充分条件。** 实际使用时无需寻找通用 Task Sink 开关，而应确认所有关键 Tensor 的 shape、算子能力、Tiling 依赖和最终执行器路径。

## 4. Tiling 下沉：再榨干一层 Host 开销

静态执行要求编译期能够生成可预分发的 Task，但**部分算子的 Tiling 依赖输入数据（`tiling_depend`）**，无法在编译期完全确定。未使用或不支持 Tiling 下沉时，这类算子可能被强制划入动态执行路径，并在运行时回到 Host 做 Tiling，带来同步和调度开销；Tiling 下沉的作用是让满足条件的算子继续留在静态下沉路径中。

### 4.1 Tiling 是什么、问题在哪

AI Core 算子执行前需要 Tiling：通常根据输入 shape、dtype 等信息把计算任务拆成可并行的「块」，并确定 `block_dim`、`tiling_key` 等执行参数；`tiling_depend` 算子还会读取指定输入的数据值。

<p align="left"><img src="./images/tiling_sink.svg" alt="Tiling 下沉对比" width="75%"></p>

对延迟敏感的高性能推理，减少 Host Tiling 与 Host-Device 同步可能降低延迟；具体收益取决于算子、模型和硬件，应以 profiling 实测为准。

### 4.2 用户怎么开启 Tiling 下沉

Tiling 下沉通过编译选项 **`ge.tiling_schedule_optimize`** 控制：

| 项 | 取值 |
| --- | --- |
| 选项名 | `ge.tiling_schedule_optimize` |
| 取值 | `"0"`（默认，关闭）/ `"1"`（开启）|

设置方式：

```shell
# 方式一：atc 离线编译，命令行参数
atc --model=model.onnx --framework=5 --output=model \
    --input_shape="input:1,3,224,224" \
    --soc_version=Ascend910B1 \
    --tiling_schedule_optimize=1
```

```cpp
// 方式二：在线编译 / Session，通过 options map 传入
std::map<ge::AscendString, ge::AscendString> options = {
    {"ge.tiling_schedule_optimize", "1"}
};
// aclgrphBuildModel(...) 或 GeSession 构造时传入
```

### 4.3 生效条件与约束

| 维度 | 要求 |
| --- | --- |
| 执行模式 | **仅静态 Shape 图生效**，动态 Shape 图不适用 |
| 算子能力 | 算子需声明支持在 AICPU 侧做 Tiling（`TILING_ON_AICPU`）|
| 硬件能力 | 设备需支持模型 Task 在线更新能力（`FEATURE_TYPE_MODEL_TASK_UPDATE`）|
| 支持产品 | Atlas A2 训练/推理、Atlas A3 训练/推理系列 |

> 约束提醒：开启 Tiling 下沉的算子不支持设置「永不超时」属性。若算子或设备能力不满足条件，编译器通常不会选择 Tiling 下沉，并可能把相应 `tiling_depend` 算子划入动态执行路径。这里不能泛化为“任何情况下都不会报错”：非法选项值、能力查询失败或其他编译异常仍可能导致失败。

## 5. 下沉收益验证：用 profiling 对比时序

模型满足下沉条件后，到底有没有收益？**不能靠感觉，要靠 profiling 数据**。验证思路是：采集执行期的 timeline，对比「Host 下发耗时」与「Device 执行耗时」的关系。

### 5.1 采集 profiling 的三种方式

| 方式 | 适用路径 | 入口 |
| --- | --- | --- |
| msprof 命令行 | 离线 ACL 应用 | `msprof --application="./your_app"` 直接拉起被测程序采集 |
| GE Options | 在线 GeSession | `GEInitializeV2` / Session 配置 `ge.exec.profilingMode="1"` + `ge.exec.profilingOptions` |
| C API 动态控制 | 在线，精确圈定采集区间 | `aclgrphProfInit → aclgrphProfCreateConfig → aclgrphProfStart → 执行 → aclgrphProfStop → aclgrphProfFinalize → aclgrphProfDestroyConfig` |

```shell
# 方式一：msprof 直接拉起离线推理程序采集（最常用于 ATC+ACL 部署验证）
msprof --application="./resnet50_infer" \
       --output=/tmp/prof_sink \
       --task-time=on
```

```cpp
// 方式二：在线 GeSession，通过 GE options 开启 profiling
std::map<ge::AscendString, ge::AscendString> config = {
    {"ge.exec.profilingMode",    "1"},
    {"ge.exec.profilingOptions", R"({"output":"/tmp/prof_sink","task_trace":"on"})"}
};
ge::GEInitializeV2(config);
```

> `PipeUtilization` 用于分析单个 AI Core Task 内计算、搬运等流水单元的耗时占比，不等同于整段执行期间的 Device 忙碌率。验证 Task Sink 时应优先观察 Host Runtime 下发事件、Device timeline 空泡和稳定态 E2E；不要预设 `PipeUtilization` 必然上升。

### 5.2 怎么从 timeline 读出「下沉是否生效、是否有收益」

采集后用 MindStudio Insight / msprof 解析出 timeline 与 summary，重点看三处：

1. **Host 侧 Runtime 下发事件数量与耗时**：
   - Host 逐算子调度：每次模型执行会看到较多与编译后 Task 对应的 Runtime Launch；受融合和 Lowering 影响，其数量不等于源码算子数。
   - 下沉调度：整图执行以**一个模型执行触发**为核心，Host 侧下发线条明显变短、变稀疏；输入输出拷贝、地址刷新等事件仍可能存在。
2. **Device 侧由 Host 引起的空泡**：下沉后这类空泡通常减少，但依赖等待、多流同步等仍可能形成间隙，不能要求所有 Task 必然首尾相接。
3. **稳定态 E2E 耗时**：在相同输入、相同 warm-up 和足够重复次数下，对比 `RunGraph` / `aclmdlExecute` 的 P50、P95 等统计值。

```
读图判据（对比下沉前后）：

          Host 下发线              Device 执行线           判断
  下沉前   ▮▮▮▮▮▮▮▮▮▮（密集）      ▯ ▯ ▯ ▯（有 Host 空泡） Host Bound，下沉有空间
  下沉后   ▮（模型执行触发）       ▮▮▮ ▮▮▮（Host 空泡减少）下沉生效；E2E 是否下降需实测
```

### 5.3 验证下沉收益的端到端场景对比

1. **基线**：用动态 Shape（或人为构造 Host Bound）路径编译并运行同一模型，固定实际输入 shape、输入数据和运行次数，完成 warm-up 后采集 timeline 与 E2E。
2. **静态场景**：用固定 Shape 编译，确认目标整图或子图进入 Task Sink；若模型包含受支持的 `tiling_depend` 算子，可再单独比较 `--tiling_schedule_optimize=0/1`。普通 Add/Relu 模型不能用于验证 Tiling 下沉。
3. **对比指标**：
   - Host 侧单次执行的 Runtime Launch 数与累计耗时（通常下降）；
   - Device timeline 中由 Host 调度引起的空泡占比（通常下降）；
   - 稳定态 E2E 的 P50 / P95（比较变化，不预设必然下降）；
   - 如需观察整体设备忙碌程度，使用对应的采样型 Device / AI Core utilization 指标，而不是把 `PipeUtilization` 当作忙碌率。

> **归因边界**：动态 Shape 与静态 Shape 对比还会同时改变 Shape 推导、内存编排、图优化及 Kernel 选择等因素，因此这是执行路径的端到端场景对比，不能把全部差异单独归因于 Task Sink。下沉收益与模型是否 Host Bound 强相关；若基线就是 Device Bound，E2E 变化不明显也是有效结论。

## 6. 小结

- 满足 Known Shape / 静态执行条件的整图或子图可使用 **Task Sink**：加载阶段一次性把编译后的 Task 预分发到 Device，执行阶段由一个模型执行 Task 触发。
- 下沉把“每次执行时多次下发编译后 Task / Kernel Launch”摊销为“加载时预分发 + 每次执行一次模型触发”，可显著降低 Host 调度开销，模型越 Host Bound，潜在收益越大。
- **静态 Shape 是核心前提但不是充分条件**：固定输入维度后，还要结合中间 Tensor shape、Tiling 依赖、算子能力和图划分确认实际执行路径；动态图中的 Known Shape 子图仍可能使用静态执行器。
- **Tiling 下沉**（`ge.tiling_schedule_optimize=1`）可把满足条件的 `tiling_depend` 算子 Tiling 从 Host 搬到 Device AICPU，并使其留在静态下沉路径中；是否生效依赖算子与硬件能力。
- 收益必须用 **profiling 验证**：对比 Host Runtime 下发事件、Device 中由 Host 引起的空泡和稳定态 E2E；`PipeUtilization` 不是整段 Device 忙碌率。

> 下一节我们看另一面：当 shape 在运行时才能确定时，GE 如何执行 Unknown Shape 部分、如何与 Known Shape 子图协同，以及 Host 侧开销从何而来。

## 课后练习

完成下列题目自测，如有错误建议结合本节对应小节复盘。

1. （判断题）静态 Shape 模型能整图下沉的根本前提是所有 Tensor 的 shape 在编译期完全确定。

2. （判断题）开启整图下沉后，每次模型执行 Host 都要把图中所有算子重新遍历下发一遍。

3. （判断题）只要模型中存在动态 Shape 节点，模型内所有子图都只能走 Host 逐算子调度，任何 Known Shape 子图都不能使用 Task Sink。

4. （单选题）模型下沉调度分为哪两个阶段？
    A. 图准备和图拆分
    B. 模型加载和模型执行
    C. InferShape 和 Tiling
    D. 编译和反编译

5. （单选题）整图下沉相比 Host 调度，性能收益最大的场景是？
    A. 模型是 Device Bound（算子计算很重）
    B. 模型是 Host Bound（单算子很快、算子很多，Host 下发是瓶颈）
    C. 模型只有一个算子
    D. 模型输入 shape 每次都在变化

6. （单选题）开启 Tiling 下沉的编译选项是哪一个？
    A. `ge.exec.reuseZeroCopyMemory`
    B. `ge.tiling_schedule_optimize`
    C. `ge.enableSingleStream`
    D. `ge.dynamicDims`

7. （多选题）以下关于下沉收益验证的描述，哪些是正确的？
    A. 可以用 `msprof --application` 直接拉起离线推理程序采集 timeline
    B. 对整图 Task Sink 路径，每次执行以一个模型执行 Task 触发预分发的任务链
    C. 下沉通常能减少由 Host 调度不及时造成的 Device 空泡，但实际收益仍需 profiling 验证
    D. 只要开启下沉，任何模型的 E2E 耗时都必然大幅下降

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/04.02_answer.txt